# Decrypt the Transaction Dataset

We now need the key to decrypt the transaction dataset, but the key is stored remotely in a Key Management Service (KMS). In a preparation step, the decryption key way stored inside the KMS and we now access it used CoCo's "Sealed Secret" feature. The identity of the Sealed Secret is defined inside the pod's workload yaml file. Like normal Kubernetes secrets, Sealed Secrets are orchestrated by the control plane and are transparently provisioned to the CoCo pod after sucessful Trustee-based **attestation**.

Such an attestation is only succesful, when the software stack is as expected and the CoCo pod is protected by Intel TDX hardware. Intel TDX makes sure that all data being loaded in the memory is encrypted, so that if an attacker tries to do a physical/virtual memory dump, the output will only be encrypted/zeroed blobs of memory. As a result, we can be certain that an attacker cannot access the transaction while they are processed by the model - providing **data in use protection**.

Let's now decrypt the transactions dataset using the automatically deployed decryption key:

In [ ]:
! openssl enc -d -aes-256-cfb -pbkdf2 -kfile /sealed/decryption/key -in transactions_encrypted -out transactions_plaintext
! echo "Data decrypted"

Checking back in our Trustee running in the secure environment, we will see that attestation happened and the key was successfully sent to this notebook:

```
2026-05-11T14:51:09.525192Z  INFO actix_web::middleware::logger: 10.128.0.2 "GET /kbs/v0/resource/default/security-policy/secure HTTP/1.1" 200 490 "-" "attestation-agent-kbs-client/0.1.0" 0.001617
2026-05-11T14:51:33.797113Z  INFO actix_web::middleware::logger: 10.128.0.2 "GET /kbs/v0/resource/default/fraud-dataset/dataset_key HTTP/1.1" 200 434 "-" "attestation-agent-kbs-client/0.1.0" 0.000883
```

Now let's see if we can read the content of the dataset

In [ ]:
! head -n 5 transactions_plaintext

We have successfully managed to decrypt the dataset!